# 12 · Files & `pathlib`

**Track 2 begins.** Data engineering is, first of all, moving bytes in and out
of files. This notebook covers reading and writing files, handling paths
portably with `pathlib`, encodings, and streaming files too big for memory.

In [ ]:
# ▶ Run this first. Locates the sample data no matter where the kernel starts.
from pathlib import Path

def find_data() -> Path:
    here = Path.cwd()
    for base in (here, *here.parents):
        if (base / 'data' / 'raw').exists():
            return base / 'data'
    raise FileNotFoundError('Run: uv run python data/build_data.py')

DATA = find_data()
RAW = DATA / 'raw'
print('Data directory:', DATA)
print('Raw files:', sorted(p.name for p in RAW.glob('*')))

## `pathlib.Path` — portable paths

Never build paths by gluing strings with `/` or `\`. `Path` joins with the `/`
operator and works identically on Windows, macOS and Linux. It also answers
questions about files.

In [ ]:
customers = RAW / 'customers.csv'      # join with /
print('path:', customers)
print('name:', customers.name)
print('stem:', customers.stem)
print('suffix:', customers.suffix)
print('parent:', customers.parent.name)
print('exists:', customers.exists())
print('size (bytes):', customers.stat().st_size)

## Reading text

For small files, `read_text()` returns the whole content as one string;
`.splitlines()` breaks it into lines. **Always specify `encoding='utf-8'`** — the
default varies by platform and causes subtle bugs.

In [ ]:
text = customers.read_text(encoding='utf-8')
lines = text.splitlines()
print('total lines:', len(lines))
print('header:', lines[0])
print('first row:', lines[1])

## Streaming line by line (memory-safe)

Opening a file gives an iterator of lines — you can process a multi-gigabyte
file while holding one line at a time. This is the default way to read big data
files in plain Python.

In [ ]:
count = 0
sample = []
with open(customers, encoding='utf-8') as f:
    header = next(f)                 # consume header line
    for line in f:                   # streams; constant memory
        count += 1
        if count <= 3:
            sample.append(line.strip())
print('data rows:', count)
print('sample:', sample)

## Writing files

Open with mode `'w'` (overwrite) or `'a'` (append). Writing through a `with`
block guarantees the file is flushed and closed. Here we write a small report
into a scratch folder next to the data.

In [ ]:
out_dir = DATA / 'staging'
out_dir.mkdir(parents=True, exist_ok=True)   # create if missing
report = out_dir / 'row_counts.txt'

with open(report, 'w', encoding='utf-8') as f:
    f.write('file,rows\n')
    f.write(f'customers,{count}\n')

print(report.read_text(encoding='utf-8'))

## Text vs bytes, and encodings

Text mode decodes bytes to `str` using an encoding; binary mode (`'rb'`/`'wb'`)
gives raw `bytes`. You need binary mode for images, Parquet, gzip, etc. Knowing
the layer prevents `UnicodeDecodeError` surprises.

In [ ]:
raw_bytes = customers.read_bytes()[:40]
print('bytes:', raw_bytes)
print('decoded:', raw_bytes.decode('utf-8'))

# encode a str to bytes explicitly
print('etl'.encode('utf-8'))

## Finding files with `glob`

`Path.glob` and `rglob` (recursive) find files by pattern — essential when
ingesting a folder of daily drops.

In [ ]:
print('csv files in raw/:')
for p in sorted(RAW.glob('*.csv')):
    print(' ', p.name, f'({p.stat().st_size} bytes)')

### Recap

`Path` joins with `/` and inspects files; `read_text`/`read_bytes` for small
files, line iteration for big ones; always set `encoding='utf-8'`; `mkdir`,
write modes, and `glob` for real ingestion. Next: structured CSV parsing.